In [1]:
import requests
import csv
import time
from pathlib import Path

# === Configuration ===
BASE_URL_SUBMISSIONS = "https://api.pullpush.io/reddit/search/submission/"
BASE_URL_COMMENTS    = "https://api.pullpush.io/reddit/search/comment/"

# January 2020
AFTER  = 1577836800  # Jan 1, 2020
BEFORE = 1580515200  # Feb 1, 2020

POSTS_PER_STATE    = 200
COMMENTS_PER_STATE = 200
REQUEST_DELAY      = 1.5

# Output
OUTPUT_DIR  = Path(r"D:\Shikha\bias_project\corpus\reddit\2020-01")
POSTS_CSV   = OUTPUT_DIR / "posts.csv"
COMMENTS_CSV = OUTPUT_DIR / "comments.csv"
ERRORS_LOG  = OUTPUT_DIR / "errors.log"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# All 50 states — subreddit name : abbreviation
STATES = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR',
    'california': 'CA', 'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID',
    'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS',
    'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS',
    'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV',
    'newhampshire': 'NH', 'newjersey': 'NJ', 'newmexico': 'NM', 'newyork': 'NY',
    'northcarolina': 'NC', 'northdakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK',
    'oregon': 'OR', 'pennsylvania': 'PA', 'rhodeisland': 'RI', 'southcarolina': 'SC',
    'southdakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT',
    'vermont': 'VT', 'virginia': 'VA', 'washington': 'WA', 'westvirginia': 'WV',
    'wisconsin': 'WI', 'wyoming': 'WY'
}

print(f"Output directory : {OUTPUT_DIR}")
print(f"Output exists    : {OUTPUT_DIR.exists()}")
print(f"States to extract: {len(STATES)}")
print(f"Period           : Jan 2020")
print(f"Cap per state    : {POSTS_PER_STATE} posts + {COMMENTS_PER_STATE} comments")

Output directory : D:\Shikha\bias_project\corpus\reddit\2020-01
Output exists    : True
States to extract: 50
Period           : Jan 2020
Cap per state    : 200 posts + 200 comments


In [2]:
def fetch_submissions(subreddit, after, before, size=200):
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": min(size, 100),
        "sort": "score",
        "sort_type": "score"
    }
    for attempt in range(3):
        try:
            r = requests.get(BASE_URL_SUBMISSIONS, params=params, timeout=30)
            if r.status_code == 200:
                return r.json().get('data', [])
            elif r.status_code == 429:
                time.sleep(30)
        except Exception as e:
            time.sleep(5)
    return []

def fetch_comments(subreddit, after, before, size=200):
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": min(size, 100),
    }
    for attempt in range(3):
        try:
            r = requests.get(BASE_URL_COMMENTS, params=params, timeout=30)
            if r.status_code == 200:
                return r.json().get('data', [])
            elif r.status_code == 429:
                time.sleep(30)
        except Exception as e:
            time.sleep(5)
    return []

# Open output CSVs
post_fields    = ['state', 'state_abbr', 'subreddit', 'post_id', 
                  'title', 'selftext', 'score', 'created_utc', 'text_length']
comment_fields = ['state', 'state_abbr', 'subreddit', 'comment_id',
                  'body', 'score', 'created_utc', 'text_length']

pf = open(POSTS_CSV,    'w', newline='', encoding='utf-8')
cf = open(COMMENTS_CSV, 'w', newline='', encoding='utf-8')
ef = open(ERRORS_LOG,   'w', encoding='utf-8')

pw = csv.DictWriter(pf, fieldnames=post_fields)
cw = csv.DictWriter(cf, fieldnames=comment_fields)
pw.writeheader()
cw.writeheader()

stats = {'states_done': 0, 'posts': 0, 'comments': 0, 'errors': 0}

for subreddit, abbr in STATES.items():
    print(f"[{abbr}] fetching...", end=' ')

    # Fetch submissions
    posts = fetch_submissions(subreddit, AFTER, BEFORE, POSTS_PER_STATE)
    time.sleep(REQUEST_DELAY)

    post_count = 0
    for p in posts:
        text = ((p.get('title') or '') + ' ' + (p.get('selftext') or '')).strip()
        if text and p.get('selftext') not in ('[deleted]', '[removed]'):
            pw.writerow({
                'state':       subreddit,
                'state_abbr':  abbr,
                'subreddit':   p.get('subreddit', subreddit),
                'post_id':     p.get('id', ''),
                'title':       p.get('title', ''),
                'selftext':    p.get('selftext', ''),
                'score':       p.get('score', 0),
                'created_utc': p.get('created_utc', ''),
                'text_length': len(text)
            })
            post_count += 1
            stats['posts'] += 1

    # Fetch comments
    comments = fetch_comments(subreddit, AFTER, BEFORE, COMMENTS_PER_STATE)
    time.sleep(REQUEST_DELAY)

    comment_count = 0
    for c in comments:
        body = (c.get('body') or '').strip()
        if body and body not in ('[deleted]', '[removed]'):
            cw.writerow({
                'state':       subreddit,
                'state_abbr':  abbr,
                'subreddit':   c.get('subreddit', subreddit),
                'comment_id':  c.get('id', ''),
                'body':        body,
                'score':       c.get('score', 0),
                'created_utc': c.get('created_utc', ''),
                'text_length': len(body)
            })
            comment_count += 1
            stats['comments'] += 1

    stats['states_done'] += 1
    print(f"{post_count} posts, {comment_count} comments")

pf.close()
cf.close()
ef.close()

print(f"\n--- Done ---")
print(f"States completed : {stats['states_done']}/50")
print(f"Total posts      : {stats['posts']}")
print(f"Total comments   : {stats['comments']}")
print(f"Posts CSV        : {POSTS_CSV}")
print(f"Comments CSV     : {COMMENTS_CSV}")

[AL] fetching... 93 posts, 84 comments
[AK] fetching... 98 posts, 87 comments
[AZ] fetching... 100 posts, 95 comments
[AR] fetching... 96 posts, 85 comments
[CA] fetching... 97 posts, 79 comments
[CO] fetching... 98 posts, 74 comments
[CT] fetching... 99 posts, 92 comments
[DE] fetching... 97 posts, 93 comments
[FL] fetching... 95 posts, 74 comments
[GA] fetching... 74 posts, 95 comments
[HI] fetching... 97 posts, 92 comments
[ID] fetching... 76 posts, 96 comments
[IL] fetching... 92 posts, 97 comments
[IN] fetching... 90 posts, 95 comments
[IA] fetching... 95 posts, 97 comments
[KS] fetching... 98 posts, 92 comments
[KY] fetching... 95 posts, 98 comments
[LA] fetching... 92 posts, 97 comments
[ME] fetching... 96 posts, 96 comments
[MD] fetching... 98 posts, 93 comments
[MA] fetching... 91 posts, 100 comments
[MI] fetching... 97 posts, 96 comments
[MN] fetching... 98 posts, 96 comments
[MS] fetching... 60 posts, 96 comments
[MO] fetching... 91 posts, 90 comments
[MT] fetching... 98 pos